In [2]:
import sys
sys.path.insert(0, '..')



In [5]:
B = 2

d_ids = ["q1", "q2"]

_d = {
    "q1": "Rome is the capital of Italy. Roma is the most beautiful city in Europe.",
}

relations = {
    "p1": "capital_of",
    "p2": "most_beautiful_city_in",
}

rel2idx = {
    "p1": 0,
    "p2": 1,
}

golden_triples = [
    # for q1
    [
        ((0,0), "p1", (5,5)),
        ((0,0), "p2", (13,13)),
        ((6,6), "p1", (5,5)),
        ((6,6), "p2", (13,13)),
    ],
]







In [4]:
from collections import defaultdict


max_s = 1
for triples in golden_triples:
    span_slot = defaultdict(int)
    for _, r, _ in triples:
        span_slot[r] += 1
        max_s = max(max_s, span_slot[r])
print(max_s)


2


In [7]:
import torch

L =15
R = 2
S = 2

gold_fwd_tail_start = torch.zeros(B, R, S, L, dtype=torch.float32)
gold_fwd_tail_end = torch.zeros(B, R, S, L, dtype=torch.float32)


for b, triples in enumerate(golden_triples):
    span_slot = defaultdict(int)
    for (hs, he), r, (ts, te) in triples:
        print(f"*******")
        print(f"Triple: {(hs, he), r, (ts, te)}")
        r_idx = rel2idx[r]
        key= (hs, he, r_idx)
        print(f"key: {key}, span_slot[key]: {span_slot[key]}")
        slot = span_slot[key]
        if slot >= S:
            continue
        if ts < L:
            gold_fwd_tail_start[b, r_idx, slot, ts] = 1.0
        if te < L:
            gold_fwd_tail_end[b, r_idx, slot, te] = 1.0
        span_slot[key] += 1

# print(f"gold_fwd_tail_start: {gold_fwd_tail_start}")
# print(f"gold_fwd_tail_end: {gold_fwd_tail_end}")


print("Gold labels: ")
for b in range(B):
    print(f"======= b={b} =======")
    for r_idx in range(R):
        print(f"\tRelation: {r_idx}")
        for s in range(S):
            print(f"\t\tSubject: {s}")
            for l in range(L):
                if gold_fwd_tail_start[b, r_idx, s, l] == 1.0:
                    print(f"\t\t\tTail start at position {l}")
                if gold_fwd_tail_end[b, r_idx, s, l] == 1.0:
                    print(f"\t\t\tTail end at position {l}")




*******
Triple: ((0, 0), 'p1', (5, 5))
key: (0, 0, 0), span_slot[key]: 0
*******
Triple: ((0, 0), 'p2', (13, 13))
key: (0, 0, 1), span_slot[key]: 0
*******
Triple: ((6, 6), 'p1', (5, 5))
key: (6, 6, 0), span_slot[key]: 0
*******
Triple: ((6, 6), 'p2', (13, 13))
key: (6, 6, 1), span_slot[key]: 0
Gold labels: 
======= b=0 =======
	Relation: 0
		Subject: 0
			Tail start at position 5
			Tail end at position 5
		Subject: 1
	Relation: 1
		Subject: 0
			Tail start at position 13
			Tail end at position 13
		Subject: 1
======= b=1 =======
	Relation: 0
		Subject: 0
		Subject: 1
	Relation: 1
		Subject: 0
		Subject: 1


In [9]:
sk_mask = torch.zeros(B, R, S, dtype=torch.float32)
sk = torch.zeros(B, R, S, 2, dtype=torch.long)
for b, triples in enumerate(golden_triples):
    slot = defaultdict(int)
    for (hs, he) , r, _ in triples:
        r_idx = rel2idx[r]
        s = slot[(hs, he, r_idx)]
        if s >= S:
            continue
        sk[b, r_idx, s, 0] = hs
        sk[b, r_idx, s, 1] = he
        sk_mask[b, r_idx, s] = 1.0
        slot[(hs, he, r_idx)] += 1




In [ ]:
print("Gold sk: ")
for b in range(B):
    print(f"======= b={b} =======")
    for r_idx in range(R):
        print(f"\tRelation: {r_idx}")
        for s in range(S):
            print(f"\t\tSubject: {s}")
            if sk_mask[b, r_idx, s] == 1.0:
                print(f"\t\t1 at position {sk[b, r_idx, s]}")


Gold sk: 
======= b=0 =======
	Relation: 0
		Subject: 0
1 at position tensor([6, 6])
		Subject: 1
	Relation: 1
		Subject: 0
1 at position tensor([6, 6])
		Subject: 1
======= b=1 =======
	Relation: 0
		Subject: 0
		Subject: 1
	Relation: 1
		Subject: 0
		Subject: 1
